In [1]:
"""
RAG-based PDF Analyzer and Ranker
================================
This tool enables you to analyze multiple PDF documents, query them with natural language, 
and rank them based on their relevance to your questions.
"""
import os
import glob
import numpy as np
import pandas as pd
from typing import List, Dict, Any, Tuple
from pathlib import Path
from dotenv import load_dotenv
# PDF processing
from PyPDF2 import PdfReader
# Vector database
from langchain_community.vectorstores import FAISS
from langchain.docstore.document import Document
# Embeddings
from langchain_openai import OpenAIEmbeddings
# LLM
from langchain_openai import ChatOpenAI

# Text splitting
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Prompt templates - FIXED: Only import once from langchain_core
from langchain_core.prompts import PromptTemplate

class PDFAnalyzer:
    """Main class for the PDF analysis and ranking system."""
    
    def __init__(self, 
                 pdf_directory: str,
                 openai_api_key: str,
                 chunk_size: int = 1000,
                 chunk_overlap: int = 100):
        """
        Initialize the PDF Analyzer.
        
        Args:
            pdf_directory: Directory containing the PDF files
            openai_api_key: OpenAI API key for embeddings and LLM
            chunk_size: Size of text chunks for processing
            chunk_overlap: Overlap between consecutive chunks
        """
        self.pdf_directory = pdf_directory
        self.openai_api_key = openai_api_key
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        
        # LLM and embeddings setup
        self.embeddings = OpenAIEmbeddings(api_key=openai_api_key)
        self.llm = ChatOpenAI(api_key=openai_api_key, model="gpt-4o")
        
        # Storage for document metadata
        self.doc_metadata = {}
        
        # Initialize vector database
        self.vector_store = None

    def load_pdfs(self) -> List[Document]:
        """
        Load all PDFs from the specified directory and convert them to Document objects.
        
        Returns:
            List of Document objects
        """
        all_docs = []
        pdf_files = glob.glob(os.path.join(self.pdf_directory, "*.pdf"))
        print(pdf_files)
        
        for pdf_path in pdf_files:
            try:
                pdf_name = os.path.basename(pdf_path)
                reader = PdfReader(pdf_path)
                text_content = ""
                
                # Extract text from each page
                for i, page in enumerate(reader.pages):
                    text_content += page.extract_text() + "\n"
                
                # Store metadata for future reference
                self.doc_metadata[pdf_name] = {
                    "path": pdf_path,
                    "pages": len(reader.pages),
                    "vendor": self._extract_vendor_info(pdf_name)
                }
                
                # Create a Document object
                doc = Document(
                    page_content=text_content,
                    metadata={"source": pdf_name, "path": pdf_path}
                )
                all_docs.append(doc)
                print(f"Loaded {pdf_name} - {len(reader.pages)} pages")
                
            except Exception as e:
                print(f"Error loading {pdf_path}: {e}")
        
        print(f"Successfully loaded {len(all_docs)} PDF documents")
        return all_docs
    
    def _extract_vendor_info(self, pdf_name: str) -> str:
        """Extract vendor information from the PDF filename."""
        # Implement vendor extraction logic based on your naming convention
        # This is a simple example - adjust based on your actual filenames
        vendor = pdf_name.split('_')[0] if '_' in pdf_name else "Unknown"
        return vendor
    
    def process_documents(self, documents: List[Document]) -> None:
        """
        Process documents by splitting them into chunks and creating a vector store.
        
        Args:
            documents: List of Document objects
        """
        # Split documents into chunks
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap
        )
        chunks = text_splitter.split_documents(documents)
        print(f"Split documents into {len(chunks)} chunks")
        
        # Create vector store from chunks
        self.vector_store = FAISS.from_documents(chunks, self.embeddings)
        print("Vector store created successfully")
    
    def query_documents(self, query: str, top_k: int = 5) -> List[Dict[str, Any]]:
        """
        Query the vector store with a natural language question.
        
        Args:
            query: Natural language query
            top_k: Number of top results to return
            
        Returns:
            List of retrieved document chunks with scores
        """
        if not self.vector_store:
            raise ValueError("Vector store not initialized. Run process_documents first.")
        
        # Search for relevant documents
        retrieval_results = self.vector_store.similarity_search_with_score(query, k=top_k)
        
        # Format results
        results = []
        for doc, score in retrieval_results:
            results.append({
                "content": doc.page_content,
                "source": doc.metadata["source"],
                "score": float(score),  # Convert numpy float to Python float
                "metadata": self.doc_metadata.get(doc.metadata["source"], {})
            })
        
        return results
    
    def rank_documents(self, query: str, criteria: List[str]) -> pd.DataFrame:
        """
        Rank all documents based on a query and specific criteria.
        
        Args:
            query: The main query to analyze documents against
            criteria: List of criteria for ranking
            
        Returns:
            DataFrame with ranked documents
        """
        if not self.vector_store:
            raise ValueError("Vector store not initialized. Run process_documents first.")
        
        results = []
        
        # Process each document and criteria
        for doc_name, metadata in self.doc_metadata.items():
            # Query the document for relevant sections
            doc_query = f"Regarding {doc_name}: {query}"
            relevant_chunks = self.query_documents(doc_query, top_k=3)
            
            # Create context from relevant chunks
            context = "\n".join([chunk["content"] for chunk in relevant_chunks])
            
            # For each criteria, evaluate the document
            criteria_scores = {}
            for criterion in criteria:
                score, explanation = self._evaluate_criterion(context, criterion, doc_name)
                criteria_scores[f"{criterion}_score"] = score
                criteria_scores[f"{criterion}_explanation"] = explanation
            
            # Calculate overall score (average of all criteria)
            criteria_score_values = [v for k, v in criteria_scores.items() if k.endswith('_score')]
            overall_score = sum(criteria_score_values) / len(criteria_score_values) if criteria_score_values else 0
            
            # Add to results
            results.append({
                "document": doc_name,
                "vendor": metadata["vendor"],
                "overall_score": overall_score,
                **criteria_scores
            })
        
        # Convert to DataFrame and sort by overall score
        df = pd.DataFrame(results)
        df = df.sort_values("overall_score", ascending=False).reset_index(drop=True)
        
        return df
    
    def _evaluate_criterion(self, context: str, criterion: str, doc_name: str) -> Tuple[float, str]:
        """
        Evaluate a document against a specific criterion.
        
        Args:
            context: Relevant document content
            criterion: The criterion to evaluate
            doc_name: Document name
            
        Returns:
            Tuple of (score, explanation)
        """
        # Create a prompt for the LLM
        prompt = PromptTemplate(
            input_variables=["context", "criterion", "doc_name"],
            template="""
            You are an expert document evaluator. Based on the following document content, 
            evaluate the document against the given criterion. Provide a score from 0-10 
            (where 10 is best) and a brief explanation.
            
            Document: {doc_name}
            
            Document Content:
            {context}
            
            Criterion to evaluate: {criterion}
            
            Respond in the format:
            SCORE: [numeric score]
            EXPLANATION: [brief explanation]
            """
        )
        
        # FIXED: Use modern LCEL approach instead of deprecated LLMChain
        chain = prompt | self.llm
        response = chain.invoke({"context": context, "criterion": criterion, "doc_name": doc_name})
        result = response.content.strip()
        
        # Parse the response
        try:
            score_line = [line for line in result.split('\n') if line.startswith('SCORE:')][0]
            score = float(score_line.replace('SCORE:', '').strip())
            
            explanation_lines = [line for line in result.split('\n') if line.startswith('EXPLANATION:')]
            explanation = explanation_lines[0].replace('EXPLANATION:', '').strip() if explanation_lines else "No explanation provided"
            
            return score, explanation
        except (IndexError, ValueError):
            # Fallback if parsing fails
            return 0.0, "Failed to evaluate criterion"
    
    def generate_summary(self, ranked_docs: pd.DataFrame, top_n: int = 5) -> str:
        """
        Generate a summary of the top ranked documents.
        
        Args:
            ranked_docs: DataFrame of ranked documents
            top_n: Number of top documents to include in summary
            
        Returns:
            Summary text
        """
        top_docs = ranked_docs.head(top_n)
        
        # Create a prompt for the LLM
        prompt = PromptTemplate(
            input_variables=["top_docs"],
            template="""
            You are an expert document analyst. Based on the following ranking data of documents,
            generate a comprehensive summary that highlights:
            
            1. Key strengths of the top-ranked documents
            2. Notable patterns across vendors
            3. Significant differences between high and low-ranked documents
            4. Recommendations based on this analysis
            
            Ranking data:
            {top_docs}
            
            Provide a concise yet informative executive summary.
            """
        )
        
        # FIXED: Use modern LCEL approach instead of deprecated LLMChain
        chain = prompt | self.llm
        response = chain.invoke({"top_docs": top_docs.to_string()})
        
        return response.content.strip()


# Example usage
def main():
    # Load environment variables
    load_dotenv(r"C:\Users\Xiaohong1\Desktop\Agents\AgenticAI_Andrew_Week2B\dot.env")
    openai_api_key = os.getenv('OPENAI_API_KEY')
    
    if not openai_api_key:
        raise ValueError("OPENAI_API_KEY not found in environment variables")
        
    # Set the directory containing your PDFs
    pdf_directory = r"C:\Users\Xiaohong1\Desktop\PDFtesting"
    
    # Initialize the analyzer
    analyzer = PDFAnalyzer(pdf_directory, openai_api_key)
    
    # Load and process documents
    documents = analyzer.load_pdfs()
    analyzer.process_documents(documents)
    
    # Define your evaluation criteria
    criteria = [
        "Program Management",
        "Reporting",
        "Enrollment and Revenue",
        "Implementation simplicity and Integration flexibility",
        "Support and maintenance",
        "Students engagement and experience",
        "Database administration and system administration",
        "Reporting and Analytics"
    ]
    
    # Rank documents based on a specific query
    query = "Which vendor provides the best data analytics solution with good scalability?"
    ranked_docs = analyzer.rank_documents(query, criteria)
    
    # Print the rankings
    print("\nDocument Rankings:")
    print(ranked_docs[["document", "vendor", "overall_score"]])
    
    # Generate a summary
    summary = analyzer.generate_summary(ranked_docs)
    print("\nExecutive Summary:")
    print(summary)


if __name__ == "__main__":
    main()

['C:\\Users\\Xiaohong1\\Desktop\\PDFtesting\\Attain Partners and Salesforce 1.pdf', 'C:\\Users\\Xiaohong1\\Desktop\\PDFtesting\\Attain Partners and Salesforce Proposal 2.pdf', 'C:\\Users\\Xiaohong1\\Desktop\\PDFtesting\\Cloud for Good.pdf', 'C:\\Users\\Xiaohong1\\Desktop\\PDFtesting\\Modern Campus.pdf', 'C:\\Users\\Xiaohong1\\Desktop\\PDFtesting\\Noodle.pdf']
Loaded Attain Partners and Salesforce 1.pdf - 1 pages
Loaded Attain Partners and Salesforce Proposal 2.pdf - 89 pages
Loaded Cloud for Good.pdf - 40 pages
Loaded Modern Campus.pdf - 74 pages
Loaded Noodle.pdf - 51 pages
Successfully loaded 5 PDF documents
Split documents into 652 chunks
Vector store created successfully

Document Rankings:
                                        document   vendor  overall_score
0                              Modern Campus.pdf  Unknown          8.125
1                                     Noodle.pdf  Unknown          7.250
2           Attain Partners and Salesforce 1.pdf  Unknown          6.000
3  A